# Create Taiwan MOST GRB Awards

Grant-pattern ingest for the legacy Ministry of Science and Technology, Taiwan, `F4320322795`, from the official Government Research Bulletin (GRB) search/export API.

- Source page: https://www.grb.gov.tw/search
- Backend API root discovered from the official GRB Angular bundle: https://grbdef.stpi.niar.org.tw
- Source scope: plan-organ code `BT100` (`科技部`), the legacy MOST funder entity.
- Script prerequisite: run `scripts/local/taiwan_most_grb_to_s3.py` to create `s3://openalex-ingest/awards/taiwan_most_grb/taiwan_most_grb_projects.parquet`.
- Source probe on 2026-06-04: 177,137 GRB source rows reported for BT100. Contractor full validation aggregates annual/period rows to one award per cited native grant ID. Amounts are source-published `本期經費(千元)` values summed across annual/period rows and converted to TWD.
- Date policy: GRB publishes ROC year-month periods, not day-level dates. `start_date` and `end_date` remain NULL; `start_year` and `end_year` are populated from source periods.
- Funder details from OpenAlex API: `F4320322795`, display `Ministry of Science and Technology, Taiwan`, ROR `https://ror.org/02kv4zf79`, DOI `10.13039/501100004663`, country `TW`.


## Imports / Config

In [ ]:
FUNDER_ID = 4320322795
FUNDER_NAME = "Ministry of Science and Technology, Taiwan"
PROVENANCE = "grb_most_projects"
PRIORITY = 210
RAW_TABLE = "taiwan_most_grb_raw"
AWARDS_TABLE = "taiwan_most_grb_awards"
S3_PATH = "s3a://openalex-ingest/awards/taiwan_most_grb/taiwan_most_grb_projects.parquet"


## Step 1: Load GRB Parquet to Staging

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.taiwan_most_grb_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/taiwan_most_grb/taiwan_most_grb_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.taiwan_most_grb_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.taiwan_most_grb_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.taiwan_most_grb_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Years, and Native Keys

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(original_plan_number) AS has_original_plan_number,
    COUNT(system_number) AS has_system_number,
    COUNT(display_name) AS has_display_name,
    COUNT(description) AS has_description,
    COUNT(executing_institution) AS has_executing_institution,
    COUNT(lead_name) AS has_lead_name,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_year) AS has_start_year,
    COUNT(end_year) AS has_end_year,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(lead_name), COUNT(*)) * 100.0, 1) AS pct_lead_name
FROM openalex.awards.taiwan_most_grb_raw;


In [ ]:
%sql
-- Money-shaped column scan via information_schema. Avoid DESCRIBE-as-subquery syntax.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'taiwan_most_grb_raw'
  AND LOWER(column_name) RLIKE 'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp|經費';


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS has_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount_twd,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount_twd,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount_twd,
    SUM(TRY_CAST(amount AS DOUBLE)) AS total_amount_twd
FROM openalex.awards.taiwan_most_grb_raw;


In [ ]:
%sql
SELECT
    start_year,
    end_year,
    COUNT(*) AS awards
FROM openalex.awards.taiwan_most_grb_raw
GROUP BY start_year, end_year
ORDER BY TRY_CAST(start_year AS INT) DESC, TRY_CAST(end_year AS INT) DESC
LIMIT 30;


In [ ]:
%sql
SELECT
    source_row_count,
    COUNT(*) AS awards
FROM openalex.awards.taiwan_most_grb_raw
GROUP BY source_row_count
ORDER BY TRY_CAST(source_row_count AS INT) DESC;


In [ ]:
%sql
SELECT
    research_method,
    research_nature,
    COUNT(*) AS awards,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.taiwan_most_grb_raw
GROUP BY research_method, research_nature
ORDER BY awards DESC
LIMIT 30;


## Step 1.6: Funder Existence Fail-Fast

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE funder_id = 4320322795`. If this returns zero rows, STOP; the insert would silently emit no awards.


In [ ]:
%sql
-- Fails the cell if the Crossref-registered funder row is missing from openalex.common.funder.
SELECT CASE
    WHEN COUNT(*) = 1 THEN 'OK: Taiwan MOST funder row exists'
    ELSE raise_error('Taiwan MOST funder row missing from openalex.common.funder')
END AS funder_check
FROM openalex.common.funder
WHERE funder_id = 4320322795;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.taiwan_most_grb_awards
USING delta
AS
WITH
most_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322795
),
raw_prepared AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(start_year AS INT) AS raw_start_year,
        TRY_CAST(end_year AS INT) AS raw_end_year,
        FROM_JSON(
            co_lead_json,
            'STRUCT<given_name:STRING,family_name:STRING,orcid:STRING,role_start_year:STRING,affiliation_name:STRING,affiliation_country:STRING>'
        ) AS parsed_co_lead,
        FROM_JSON(
            investigators_json,
            'ARRAY<STRUCT<given_name:STRING,family_name:STRING,orcid:STRING,role_start_year:STRING,affiliation_name:STRING,affiliation_country:STRING>>'
        ) AS parsed_investigators
    FROM openalex.awards.taiwan_most_grb_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
),
cleaned AS (
    SELECT
        *,
        CASE WHEN raw_start_year > YEAR(current_date()) + 1 THEN NULL ELSE raw_start_year END AS parsed_start_year,
        CASE WHEN raw_start_year > YEAR(current_date()) + 1 THEN NULL ELSE raw_end_year END AS parsed_end_year
    FROM raw_prepared
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 AS id,
        g.display_name AS display_name,
        g.description AS description,
        f.funder_id,
        g.funder_award_id,
        g.parsed_amount AS amount,
        g.currency AS currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'research' AS funding_type,
        COALESCE(
            NULLIF(TRIM(g.research_method), ''),
            'MOST GRB project'
        ) AS funder_scheme,
        'grb_most_projects' AS provenance,
        CAST(NULL AS DATE) AS start_date,
        CAST(NULL AS DATE) AS end_date,
        g.parsed_start_year AS start_year,
        g.parsed_end_year AS end_year,
        struct(
            NULLIF(TRIM(g.lead_given_name), '') AS given_name,
            NULLIF(TRIM(g.lead_family_name), '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                NULLIF(TRIM(g.executing_institution), '') AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CASE
            WHEN g.parsed_co_lead IS NULL THEN CAST(NULL AS STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
            ELSE struct(
                NULLIF(TRIM(g.parsed_co_lead.given_name), '') AS given_name,
                NULLIF(TRIM(g.parsed_co_lead.family_name), '') AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    NULLIF(TRIM(g.parsed_co_lead.affiliation_name), '') AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        END AS co_lead_investigator,
        CASE
            WHEN g.parsed_investigators IS NULL OR size(g.parsed_investigators) = 0 THEN CAST(NULL AS ARRAY<STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >>)
            ELSE TRANSFORM(g.parsed_investigators, x -> struct(
                NULLIF(TRIM(x.given_name), '') AS given_name,
                NULLIF(TRIM(x.family_name), '') AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    NULLIF(TRIM(x.affiliation_name), '') AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            ))
        END AS investigators,
        g.landing_page_url AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM cleaned g
    CROSS JOIN most_funder f
)
SELECT * FROM awards_transformed;


In [ ]:
%sql
SELECT COUNT(*) AS awards_rows
FROM openalex.awards.taiwan_most_grb_awards;


In [ ]:
%sql
SELECT *
FROM openalex.awards.taiwan_most_grb_awards
LIMIT 10;


## Step 3: Delete by Provenance/Priority and Insert

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'grb_most_projects'
  AND priority = 210;


In [ ]:
%sql
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    210 AS priority
FROM openalex.awards.taiwan_most_grb_awards;


## Step 6: Verification

In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_year) AS has_start_year,
    COUNT(lead_investigator) AS has_lead_investigator,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) AS pct_start_year
FROM openalex.awards.taiwan_most_grb_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount,
    SUM(amount) AS total_amount
FROM openalex.awards.taiwan_most_grb_awards;


In [ ]:
%sql
SELECT start_year, end_year, COUNT(*) AS awards
FROM openalex.awards.taiwan_most_grb_awards
GROUP BY start_year, end_year
ORDER BY start_year DESC, end_year DESC
LIMIT 30;


In [ ]:
%sql
SELECT COUNT(*) AS future_year_rows
FROM openalex.awards.taiwan_most_grb_awards
WHERE start_year > YEAR(current_date()) + 1;


In [ ]:
%sql
SELECT funder.display_name, funder.id, funder.ror_id, funder.doi, COUNT(*) AS awards
FROM openalex.awards.taiwan_most_grb_awards
GROUP BY funder.display_name, funder.id, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT
    priority,
    provenance,
    COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'grb_most_projects'
  AND priority = 210
GROUP BY priority, provenance;


## Handoff / Admin Notes

Contractor has no S3/Databricks access. Admin must:

1. Run `scripts/local/taiwan_most_grb_to_s3.py` without `--skip-upload` to upload the full parquet.
2. Run this Databricks notebook and review Step 6 verification output.
3. Run `notebooks/awards/CreateAwards.ipynb` and downstream QA before marking the tracker Complete.

No job YAML is included in this contractor-complete PR.
